# Simulate data 
... and compress it with relative binning.

Assumes you have run the previous notebook: `1-generate_parameters.ipynb`

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
import sys
sys.path.append('../../cogwheel/')
sys.path.append('..')

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal
lal.swig_redirect_standard_output_error(False)

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cogwheel import data

from cogwheel_machine import config
from cogwheel_machine.waveform_model import PhenomenologicalWaveformGenerator
from cogwheel_machine.simulation import DataPreprocessor, Simulator, simulate_and_preprocess_samples

In [ ]:
# Assumes you have run the previous notebook: 1-generate_parameters.ipynb
sim_dir = Path('../data/set_0')
simulation_parameters = pd.read_feather(sim_dir/'simulation_parameters.feather')

In [ ]:
approximant = 'IMRPhenomD'

In [ ]:
simulator = Simulator(config.EVENT_DATA_KWARGS, approximant)

In [ ]:
dummy_event_data = data.EventData.gaussian_noise(**config.EVENT_DATA_KWARGS)
waveform_model = PhenomenologicalWaveformGenerator.from_event_data(
    event_data=dummy_event_data,
    pn_phase_tol=0.1)

data_preprocessor = DataPreprocessor(
    waveform_model,
    pn_phase_tol_compression=config.PN_PHASE_TOL_COMPRESSION
)

## Run industrially
Run over all simulation parameters in parallel

In [ ]:
# Edit `processes` if you would like to use fewer cores (None means all available)
simulation_data, folded_sampled_params, unfolding_labels = simulate_and_preprocess_samples(
    simulator,
    data_preprocessor,
    simulation_parameters,
    processes=None)

In [ ]:
np.save(sim_dir/'simulation_data.npy', simulation_data)
np.save(sim_dir/'folded_sampled_params.npy', folded_sampled_params)
np.save(sim_dir/'unfolding_labels.npy', unfolding_labels)

## Exploratory plot

In [ ]:
def get_n_fbin():
    """Get number of frequencies"""
    # TODO make this the responsibility of DataPreprocessor
    from cogwheel_machine.rbsplines import RelativeBinningSplines
    
    rb_splines = RelativeBinningSplines(
        dummy_event_data.frequencies[dummy_event_data.fslice],
        pn_phase_tol=config.PN_PHASE_TOL_COMPRESSION)
    
    return len(rb_splines.fbin)

In [ ]:
n_samples_to_plot = 100
n_fbin = get_n_fbin()
n_det = data_preprocessor.waveform_model.n_det
n_heterodyned_data = 2 * n_det * n_fbin

plt.figure()
plt.plot(simulation_data[:n_samples_to_plot, :n_heterodyned_data].T, lw=0.2, alpha=.5);
plt.ylabel(r'$\propto$ Re/Im $d e^{-i \Phi} / (S A_{\rm best})$')
plt.xlabel(rf'Monotonic with frequency in each block of {n_fbin}')